# Record the RCC relay with Qwen3-8B

This notebook runs the existing demo recorder at **W=16 and W=4**, with the same
r4 budget, questions and source documents. It preserves unedited answers,
selected-token lineage, synchronized wall times, logs and runtime metadata.
Final answers allow up to **1,024 generated tokens** for both methods.
The task joins a booking reference, changed destination and ID rule; no arithmetic
is required. Each question records an uncompressed full-context answer first.

1. Choose a **24 GB-class or larger CUDA GPU** and a **high-RAM host** under
   **Runtime > Change runtime type**. An L4 or A100 is the intended target.
   This is an unquantized BF16/FP16 run. The notebook checks GPU memory before
   downloading weights; a 16 GB T4 is too small for this configuration.
2. Upload the separate `rcc-demo-colab-source.zip` when prompted, then allow
   Drive mounting. This does not require a public repository or a GitHub token.
3. Run the remaining cells in order. Results go to a unique directory under
   `MyDrive/rcc-demo/`. The final cell downloads a portable replay ZIP.

Build the source archive from the RCC repository after committing your changes:

```bash
mkdir -p output/colab
git archive --format=zip --output=output/colab/rcc-demo-colab-source.zip HEAD
```

Colab hardware availability and session lifetime vary; see the
[Colab FAQ](https://research.google.com/colaboratory/faq.html).
The [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B) describes the model.
The demo deliberately keeps the existing non-thinking, greedy setup for both
methods; this is not a claim that greedy decoding is Qwen's recommended setting.


In [ ]:
import hashlib
import json
import logging
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch

MODEL = "Qwen/Qwen3-8B"
REVISION = "main"  # Use a recorded model commit hash to repeat that exact revision.
RATIO = 4
WIDTHS = (16, 4)
TEST_MODE = os.environ.get("RCC_DEMO_TEST_MODE") == "1"
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

if TEST_MODE:
    MODEL = os.environ["RCC_DEMO_MODEL"]
    SOURCE_ZIP = Path(os.environ["RCC_DEMO_SOURCE_ZIP"])
    OUTPUT_ROOT = Path(os.environ["RCC_DEMO_OUTPUT"])
    DEVICE = "cpu"
else:
    from google.colab import drive, files

    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime before recording.")
    gpu = torch.cuda.get_device_properties(0)
    if gpu.total_memory < 22 * 1024**3:
        raise RuntimeError("Use a 24 GB-class GPU or larger for unquantized Qwen3-8B.")
    logging.info(
        "GPU: %s (%.1f GiB). Use a high-RAM host for model loading.",
        gpu.name,
        gpu.total_memory / 1024**3,
    )
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one RCC source ZIP.")
    SOURCE_ZIP = Path(next(iter(uploaded)))
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/rcc-demo")
    DEVICE = "cuda"

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN = OUTPUT_ROOT / run_id
RUN.mkdir(parents=True)
SOURCE = Path(tempfile.mkdtemp(prefix="rcc-demo-source-"))
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in archive.namelist()):
        raise ValueError("Source ZIP contains an unsafe path.")
    archive.extractall(SOURCE)
if not (SOURCE / "demo/record.py").is_file() or not (SOURCE / "pyproject.toml").is_file():
    raise ValueError("Expected a git archive of the RCC repository root.")
shutil.copyfile(SOURCE_ZIP, RUN / "source.zip")
(RUN / "config.json").write_text(
    json.dumps(
        {
            "model": MODEL,
            "revision": REVISION,
            "device": DEVICE,
            "ratio": RATIO,
            "span_widths": WIDTHS,
            "source_sha256": hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest(),
            "test_mode": TEST_MODE,
        },
        indent=2,
    )
    + "\n"
)
logging.info("Run directory: %s", RUN)

## Install and capture the environment

The notebook uses native Hugging Face model state, matching the local demo.
It does not launch vLLM. It pins Transformers 4.57.1 and keeps Colab's installed
PyTorch. Package versions, GPU/driver details and the exact source ZIP are saved.
Commands run in subprocesses, so the notebook does not need a kernel restart.


In [ ]:
run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(SOURCE / "src") + os.pathsep + run_env.get("PYTHONPATH", "")
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env["HF_HUB_DISABLE_XET"] = "1"
if TEST_MODE:
    run_env["HF_HUB_OFFLINE"] = "1"


def run_command(command: list[str], log_name: str) -> None:
    """Save complete stdout/stderr and fail visibly when a command fails."""
    log_path = RUN / log_name
    logging.info("Running %s; log: %s", command, log_path)
    with log_path.open("w") as log:
        result = subprocess.run(
            command, cwd=SOURCE, env=run_env, stdout=log, stderr=subprocess.STDOUT, check=False
        )
    if result.returncode:
        logging.error("%s", log_path.read_text()[-4000:])
        raise RuntimeError(f"Command failed ({result.returncode}); see {log_path}")


if not TEST_MODE:
    run_command(
        [sys.executable, "-m", "pip", "install", str(SOURCE), "transformers==4.57.1"], "install.log"
    )
    run_command(["nvidia-smi"], "gpu.txt")
run_command([sys.executable, "-m", "pip", "freeze"], "packages.txt")
DEMO = RUN / "demo"
DEMO.mkdir()
for name in ("index.html", "README.md", "record.py", "colab.ipynb"):
    shutil.copyfile(SOURCE / "demo" / name, DEMO / name)

## Record both widths

Each width records a full-context answer before the text and RCLC relays for
each question. Inspect this baseline before attributing errors to communication. The model is loaded once
per width; loading is outside the reported timings. Each completed question is
checkpointed into its trace, and per-hop timing and answers are also written to
the log. CUDA is synchronized at timing boundaries. This is one observation per
method and question, not a performance benchmark.

A failed or interrupted run leaves its existing logs and completed questions
in Drive. You can still run the download cell below to collect those artifacts.
Keep failed runs; do not silently replace them with a successful rerun.


In [ ]:
for width in WIDTHS:
    trace_name = "trace.json" if width == 16 else f"trace-w{width}.json"
    run_command(
        [
            sys.executable,
            "-u",
            "demo/record.py",
            "--model",
            MODEL,
            "--revision",
            REVISION,
            "--device",
            DEVICE,
            "--ratio",
            str(RATIO),
            "--span-size",
            str(width),
            "--output",
            str(DEMO / trace_name),
        ],
        f"record-w{width}.log",
    )
    trace = json.loads((DEMO / trace_name).read_text())
    for case in trace["cases"]:
        for method in ("full_context", "text", "rclc"):
            result = case[method]
            logging.info(
                "W=%s | %s | %.3fs | %s\n%s",
                width,
                method,
                result["seconds"],
                case["question"],
                result["answer"]["text"],
            )

## Download and replay later

Run this cell even after a recording error to download partial results and logs.
Drive already holds the individual files. The ZIP contains `source.zip`, config,
package/GPU details, complete command logs, both traces when completed, and the
static viewer. It contains no model weights, credentials or environment dump.

Unzip it locally, open a terminal in the extracted folder, and run:

```bash
python -m http.server 8765 --bind 127.0.0.1
```

Then open <http://127.0.0.1:8765/demo/>. If that port is already in use, stop that
server or use another port. No inference is needed to replay
the GPU recording. Inspect both questions and both widths before choosing footage.


In [ ]:
BUNDLE = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
logging.info("Saved replay and logs: %s", BUNDLE)
if not TEST_MODE:
    files.download(str(BUNDLE))